In [ ]:
"""
common.py - shared configuration and helpers for Steps 2, 3a, 3b, 3c, 3d and the EDA.
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Keep this file next to step2_data_ingestion.py, step3a-3d, run_pipeline.py, business_rules.py, cohort_scoring.py
and eda_data_understanding_v6.py. Every path resolves from PIPELINE_ROOT (default: this file's folder), so all
scripts - including ingestion and the EDA - read and write the same data/ folder regardless of the working directory.

Change log (see decisions.md): D01 RAW/LOG/EDA/GOLD paths + used by Step 2 and EDA, D09 notebook-safe finish().
"""

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import duckdb

## 2. Paths (one layout for every script)

In [ ]:
_HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()   # .py or .ipynb
PROJECT_DIR = Path(os.getenv("PIPELINE_ROOT", _HERE)).resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"                  # 4 source CSVs (Step 2 input)
BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
SILVER_DIR = PROJECT_DIR / "data" / "silver"
QUARANTINE_DIR = PROJECT_DIR / "data" / "quarantine"
FEATURE_DIR = PROJECT_DIR / "data" / "features"
COHORT_DIR = PROJECT_DIR / "data" / "cohort"
GOLD_DIR = PROJECT_DIR / "data" / "gold"                # reserved for the production scoring step (agents)
EDA_DATA_DIR = PROJECT_DIR / "data" / "eda"             # EDA artefacts - never read by the production pipeline
REPORT_DIR = PROJECT_DIR / "reports"
CONFIG_DIR = PROJECT_DIR / "config"
LOG_DIR = PROJECT_DIR / "logs"
DB_PATH = PROJECT_DIR / "data" / "customer_scoring.duckdb"
ALLOWED_VALUES_FILE = CONFIG_DIR / "allowed_values.json"              # reviewed by you
ALLOWED_VALUES_SUGGESTED = CONFIG_DIR / "allowed_values_suggested.json"  # written by 3a

MEMORY_LIMIT = os.getenv("PIPELINE_MEMORY", "6GB")
THREADS = int(os.getenv("PIPELINE_THREADS", 4))

TABLES = ["customers", "interactions", "transactions", "engagements"]   # customers first (FK target)
EVENT_TABLES = ["interactions", "transactions", "engagements"]
PK = {"customers": "customer_id", "interactions": "interaction_id",
      "transactions": "transaction_id", "engagements": "engagement_id"}
EVENT_TS = {"interactions": "interaction_ts", "transactions": "transaction_ts",
            "engagements": "event_ts"}

## 3. Data window (1 Oct 2025 - 30 Sep 2026)

In [ ]:
WINDOW_START = "2025-10-01"
WINDOW_END = "2026-09-30"          # inclusive
WINDOW_END_EXCL = "2026-10-01"     # exclusive form used by the EDA
FIRST_MONTH, LAST_MONTH = "2025-10", "2026-09"

# Columns that must never be used as model features (label protection)
LABEL_COLUMNS = ["sentiment_label", "sentiment_score_gt"]
FREE_TEXT_COLUMNS = ["interaction_text"]
LINEAGE_COLUMNS = ["_source_file", "_ingested_at"]

## 4. Column catalogue

In [ ]:
# Categorical columns profiled for distinct values / allowed-value checks
CATEGORICAL = {
    "customers": ["gender", "region", "country", "customer_segment", "acquisition_channel",
                  "income_band", "preferred_channel", "loyalty_tier", "marketing_consent"],
    "interactions": ["channel", "interaction_type", "resolution_status",
                     "first_contact_resolution", "sentiment_label"],
    "transactions": ["transaction_type", "product_category", "currency", "payment_method",
                     "merchant_channel", "transaction_status", "is_international"],
    "engagements": ["engagement_channel", "event_type", "device_type", "is_converted"],
}

# Flag-like text columns converted to BOOLEAN in Silver
BOOLEAN_COLUMNS = {
    "customers": ["marketing_consent"],
    "interactions": ["first_contact_resolution"],
    "transactions": ["is_international"],
    "engagements": ["is_converted"],
}
TRUE_TOKENS = ["true", "t", "yes", "y", "1"]
FALSE_TOKENS = ["false", "f", "no", "n", "0"]

## 5. Helpers

In [ ]:
def log(msg: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)


def sql_path(p: Path) -> str:
    """DuckDB-safe path string (forward slashes work on Windows too)."""
    return p.as_posix().replace("'", "''")


def get_connection(read_only: bool = False, db_path: Path | None = None,
                   memory: str | None = None, threads: int | None = None) -> duckdb.DuckDBPyConnection:
    """Configured DuckDB connection. db_path lets the EDA keep its own work database."""
    db = Path(db_path or DB_PATH)
    db.parent.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(str(db), read_only=read_only)
    con.execute(f"SET memory_limit = '{memory or MEMORY_LIMIT}'")
    con.execute(f"SET threads = {threads or THREADS}")
    con.execute("SET preserve_insertion_order = false")
    tmp = PROJECT_DIR / "data" / "duckdb_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    con.execute(f"SET temp_directory = '{sql_path(tmp)}'")      # spill to disk instead of OOM
    return con


def register_layer(con: duckdb.DuckDBPyConnection, layer_dir: Path, prefix: str,
                   names: list[str] | None = None) -> None:
    """Create/refresh views <prefix>_<name> over <layer_dir>/<name>.parquet.
    Missing lineage columns are added as NULL so EXCLUDE / ORDER BY on them never fails."""
    for name in names or TABLES:
        pq = layer_dir / f"{name}.parquet"
        if not pq.exists():
            raise FileNotFoundError(f"Missing {pq} - run the previous step first.")
        src = f"read_parquet('{sql_path(pq)}')"
        have = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {src}").fetchall()}
        extra = "".join([", CAST(NULL AS VARCHAR) AS _source_file" if "_source_file" not in have else "",
                         ", CAST(NULL AS TIMESTAMP) AS _ingested_at" if "_ingested_at" not in have else ""])
        con.execute(f"CREATE OR REPLACE VIEW {prefix}_{name} AS SELECT *{extra} FROM {src}")


def present_columns(con: duckdb.DuckDBPyConnection, view: str, wanted: list[str]) -> list[str]:
    """Configured columns that actually exist in the view (optional columns are skipped, not fatal)."""
    have = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {view}").fetchall()}
    return [c for c in wanted if c in have]


def scalar(con: duckdb.DuckDBPyConnection, sql: str):
    """First value of the first row."""
    return con.execute(sql).fetchall()[0][0]


def copy_to_parquet(con: duckdb.DuckDBPyConnection, query_or_table: str, out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY ({query_or_table}) TO '{sql_path(out)}' "
                f"(FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 122880)")


def copy_to_csv(con: duckdb.DuckDBPyConnection, query: str, out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY ({query}) TO '{sql_path(out)}' (HEADER, DELIMITER ',')")


def norm(col: str) -> str:
    """SQL expression: trimmed, lower-case text with '' -> NULL (works for any input type)."""
    return f"NULLIF(lower(trim(CAST({col} AS VARCHAR))), '')"


def load_allowed_values() -> dict | None:
    if ALLOWED_VALUES_FILE.exists():
        return json.loads(ALLOWED_VALUES_FILE.read_text())
    return None


def print_checks(checks: list[dict]) -> bool:
    """Print PASS/FAIL check rows; returns True when every check passed."""
    for c in checks:
        print(f"  [{c['status']}] {c['check']:<45} violations = {c['violations']:,}")
    return all(c["status"] == "PASS" for c in checks)


def checks_from(pairs: list[tuple[str, int]]) -> list[dict]:
    """[(name, violations)] -> check dicts with PASS/FAIL status."""
    return [{"check": n, "violations": int(v or 0), "status": "PASS" if not v else "FAIL"} for n, v in pairs]

## 6. Run status (works from a terminal, VS Code Interactive and Jupyter)

In [ ]:
def in_notebook() -> bool:
    try:
        get_ipython()          # noqa: F821  (defined only inside IPython)
        return True
    except NameError:
        return False


def run_status(ok: bool) -> int:
    """Process exit code: 0 = passed, 1 = failed (run_pipeline.py stops on 1)."""
    return 0 if ok else 1


def finish(ok: bool) -> None:
    """End of a script: exit code in a terminal, a printed status in a notebook (no SystemExit traceback)."""
    if in_notebook():
        print(f"STATUS: {'PASSED' if ok else 'FAILED'}")
    else:
        sys.exit(run_status(ok))